In [3]:
import SimpleITK as sitk
import numpy as np

# -----------------------------
# 1. Load NIfTI
# -----------------------------
case_num = "4a69f761-e9c5-46ab-8ff0-13bf2f418027"
input_path = fr"C:\Users\USER\Documents\Projects\ARNA_3D\data\case_{case_num}\mask\segment__combined.nii.gz"

img = sitk.ReadImage(input_path)
arr = sitk.GetArrayFromImage(img)  # (z, y, x)

# -----------------------------
# 2. Tumor mask (label = 1)
# -----------------------------
tumor_mask = (arr == 1).astype(np.uint8)
tumor_img = sitk.GetImageFromArray(tumor_mask)
tumor_img.CopyInformation(img)

# -----------------------------
# 3. Connected Component (3D)
# -----------------------------
cc = sitk.ConnectedComponent(tumor_img)
cc = sitk.RelabelComponent(cc, sortByObjectSize=True)

cc_arr = sitk.GetArrayFromImage(cc)
num_lcc = int(cc_arr.max())
print(f"[Tumor] LCC 개수: {num_lcc}")

# -----------------------------
# 4. 각 LCC 중심 voxel 좌표
# -----------------------------
for label in range(1, num_lcc + 1):
    coords = np.argwhere(cc_arr == label)  # (z, y, x)
    centroid_zyx = coords.mean(axis=0)

    # ITK-SNAP index 순서 (x, y, z)
    centroid_xyz = centroid_zyx[::-1]

    print(f"\nLCC #{label}")
    print(f" - voxel 수: {coords.shape[0]}")
    print(f" - 중심 voxel 좌표 (x, y, z): {centroid_xyz}")


[Tumor] LCC 개수: 3

LCC #1
 - voxel 수: 22881
 - 중심 voxel 좌표 (x, y, z): [466.89091386 844.39399502  85.33460076]

LCC #2
 - voxel 수: 480
 - 중심 voxel 좌표 (x, y, z): [443.18125 906.96875 135.     ]

LCC #3
 - voxel 수: 48
 - 중심 voxel 좌표 (x, y, z): [909.5 923.5 150. ]


In [ ]:
import numpy as np
import SimpleITK as sitk
from scipy import ndimage
from dataclasses import dataclass
from typing import Literal

# 라벨 정의
LABELS = {
    "TUMOR": 1,
    "KIDNEY": 2,
    "ARTERY": 3,
    "VEIN": 4,
}

def _get_structure(connectivity: int) -> np.ndarray:
    """Connected component용 구조체 (3D)"""
    if connectivity == 6:
        return ndimage.generate_binary_structure(3, 1)
    if connectivity == 26:
        return ndimage.generate_binary_structure(3, 2)
    raise ValueError("connectivity must be 6 or 26")

@dataclass
class ComponentInfo:
    """개별 Connected Component 정보"""
    label_name: str
    component_id: int
    volume_mm3: float
    voxel_count: int
    sample_voxel: tuple[int, int, int]  # (x, y, z) 임의 voxel 인덱스
    bbox: tuple[tuple[int, int], ...]  # ((z_min, z_max), (y_min, y_max), (x_min, x_max))

@dataclass
class TumorLocationInfo:
    """종양 위치 분석 정보"""
    component_id: int
    volume_mm3: float
    sample_voxel: tuple[int, int, int]  # (x, y, z) 임의 voxel 인덱스
    location: Literal["internal", "protruding", "external"]  # 내부/돌출/외부
    overlap_ratio: float  # 라벨 겹침 비율 (참고용)
    contact_ratio: float  # 경계에서 신장과 접촉하는 비율 (0~1)
    distance_to_kidney_mm: float  # 신장까지 최소 거리 (external인 경우)


In [ ]:
def analyze_connected_components(
    volume: np.ndarray, 
    label_value: int, 
    label_name: str,
    spacing: tuple[float, float, float],
    connectivity: int = 26
) -> list[ComponentInfo]:
    """특정 라벨의 Connected Components 분석"""
    mask = (volume == label_value).astype(np.uint8)
    structure = _get_structure(connectivity)
    labeled_array, num_features = ndimage.label(mask, structure=structure)

    if num_features == 0:
        return []

    voxel_volume = spacing[0] * spacing[1] * spacing[2]

    components = []
    for comp_id in range(1, num_features + 1):
        comp_mask = (labeled_array == comp_id)
        voxel_count = int(np.sum(comp_mask))
        volume_mm3 = voxel_count * voxel_volume

        coords = np.where(comp_mask)
        z0, y0, x0 = coords[0][0], coords[1][0], coords[2][0]
        sample_voxel = (int(x0), int(y0), int(z0))

        bbox = tuple((int(coords[i].min()), int(coords[i].max())) for i in range(3))

        components.append(ComponentInfo(
            label_name=label_name,
            component_id=comp_id,
            volume_mm3=volume_mm3,
            voxel_count=voxel_count,
            sample_voxel=sample_voxel,
            bbox=bbox
        ))

    components.sort(key=lambda x: x.volume_mm3, reverse=True)
    return components


In [ ]:
def analyze_tumor_location(
    volume: np.ndarray,
    spacing: tuple[float, float, float],
    connectivity: int = 26
) -> list[TumorLocationInfo]:
    """종양의 신장 대비 위치 분석"""
    tumor_mask = (volume == LABELS["TUMOR"]).astype(np.uint8)
    kidney_mask = (volume == LABELS["KIDNEY"]).astype(np.uint8)

    structure = _get_structure(connectivity)
    tumor_labeled, num_tumors = ndimage.label(tumor_mask, structure=structure)

    if num_tumors == 0:
        return []

    voxel_volume = spacing[0] * spacing[1] * spacing[2]
    spacing_zyx = (spacing[2], spacing[1], spacing[0])
    bg_mask = ~(kidney_mask.astype(bool) | tumor_mask.astype(bool))

    if np.any(kidney_mask):
        kidney_distance = ndimage.distance_transform_edt(
            ~kidney_mask.astype(bool), sampling=spacing_zyx
        )
    else:
        kidney_distance = np.full_like(volume, fill_value=np.inf, dtype=float)

    results = []
    for tumor_id in range(1, num_tumors + 1):
        tumor_comp = (tumor_labeled == tumor_id)
        tumor_voxels = int(np.sum(tumor_comp))
        volume_mm3 = tumor_voxels * voxel_volume

        coords = np.where(tumor_comp)
        z0, y0, x0 = coords[0][0], coords[1][0], coords[2][0]
        sample_voxel = (int(x0), int(y0), int(z0))

        overlap_voxels = int(np.sum(tumor_comp & kidney_mask.astype(bool)))
        overlap_ratio = overlap_voxels / tumor_voxels if tumor_voxels > 0 else 0

        border = ndimage.binary_dilation(tumor_comp, structure=structure) & ~tumor_comp
        kidney_contact = int(np.sum(border & kidney_mask.astype(bool)))
        bg_contact = int(np.sum(border & bg_mask))
        contact_ratio = kidney_contact / max(kidney_contact + bg_contact, 1)
        touches_kidney = (kidney_contact > 0) or (overlap_voxels > 0)
        touches_background = bg_contact > 0

        if touches_kidney and not touches_background:
            location, distance = "internal", 0.0
        elif touches_kidney and touches_background:
            location, distance = "protruding", 0.0
        else:
            location = "external"
            distance = float(np.min(kidney_distance[tumor_comp]))

        results.append(TumorLocationInfo(
            component_id=tumor_id,
            volume_mm3=volume_mm3,
            sample_voxel=sample_voxel,
            location=location,
            overlap_ratio=overlap_ratio,
            contact_ratio=contact_ratio,
            distance_to_kidney_mm=distance
        ))

    results.sort(key=lambda x: x.volume_mm3, reverse=True)
    return results


In [ ]:
def print_analysis_report(
    all_components: dict[str, list[ComponentInfo]],
    tumor_locations: list[TumorLocationInfo]
):
    """분석 결과 출력"""
    print("=" * 72)
    print("Segmentation Analysis Report")
    print("=" * 72)

    for label_name, components in all_components.items():
        print(f"\n{'-' * 60}")
        print(f"[{label_name}]")
        print(f"{'-' * 60}")
        print(f"  Components: {len(components)}")

        if components:
            total_volume = sum(c.volume_mm3 for c in components)
            print(f"  Total Volume: {total_volume:,.2f} mm^3 ({total_volume/1000:.2f} cm^3)")
            print("\n  Details:")
            for i, comp in enumerate(components, start=1):
                x, y, z = comp.sample_voxel
                print(f"    #{i:02d} | Volume: {comp.volume_mm3:,.2f} mm^3 ({comp.volume_mm3/1000:.2f} cm^3)")
                print(f"          Voxels: {comp.voxel_count:,}")
                print(f"          Sample Voxel (x, y, z): ({x}, {y}, {z})")

    if tumor_locations:
        print(f"\n{'=' * 72}")
        print("Tumor Location vs Kidney")
        print(f"{'=' * 72}")

        location_names = {
            "internal": "내부 (신장 안에 완전히 위치)",
            "protruding": "돌출 (신장에서 바깥으로 나옴)",
            "external": "외부 (신장과 완전히 분리)"
        }

        for i, tumor in enumerate(tumor_locations, start=1):
            x, y, z = tumor.sample_voxel
            print(f"\n  Tumor #{i:02d}")
            print(f"    Volume: {tumor.volume_mm3:,.2f} mm^3 ({tumor.volume_mm3/1000:.2f} cm^3)")
            print(f"    Sample Voxel (x, y, z): ({x}, {y}, {z})")
            print(f"    Location: {location_names[tumor.location]}")
            print(f"    Kidney Contact Ratio: {tumor.contact_ratio*100:.1f}%")
            print(f"    Label Overlap Ratio (ref): {tumor.overlap_ratio*100:.1f}%")
            if tumor.location == "external":
                print(f"    Min Distance to Kidney: {tumor.distance_to_kidney_mm:.2f} mm")

    print(f"\n{'=' * 72}")


In [ ]:
# NIfTI 파일 로드
case_num = "4a69f761-e9c5-46ab-8ff0-13bf2f418027"
input_path = fr"C:\Users\USER\Documents\Projects\ARNA_3D\data\case_{case_num}\mask\segment__combined.nii.gz"
image = sitk.ReadImage(input_path)
volume = sitk.GetArrayFromImage(image)  # (z, y, x) 순서
spacing = image.GetSpacing()  # (x, y, z)

print(f"Input: {input_path}")
print(f"Volume shape (z, y, x): {volume.shape}")
print(f"Spacing (x, y, z): {spacing} mm")
print(f"Labels present: {np.unique(volume)}")


In [ ]:
# 각 구조별 Connected Components 분석
connectivity = 26  # 6 또는 26
all_components = {}
for label_name, label_value in LABELS.items():
    components = analyze_connected_components(volume, label_value, label_name, spacing, connectivity)
    all_components[label_name] = components

# 종양 위치 분석 (내부/돌출/외부)
tumor_locations = analyze_tumor_location(volume, spacing, connectivity)

# 결과 출력 (가독성 강화)
print_analysis_report(all_components, tumor_locations)
